In [ ]:
import numpy as np
import sys
import os
import nibabel as nib
sys.path.append(os.getcwd().removesuffix('\\evaluation'))
print(sys.path)

from models.tooth_fairy_1 import build_hyspark_tooth_fairy_1
from models.class_segmentor_model import build_class_segmentor_model
import torch as tc
import torch.nn as nn
from evaluation.evaluate_agh_segmentor import evaluate_agh

model =build_class_segmentor_model(inchannels=1, outchannels=34, depths=[16, 32, 64, 128, 192])
model.load_state_dict(tc.load('../models/weights/agh_segmentor_best.pt', weights_only=True))
file = '../data/snippets_img/1.npy'

image = np.load(file).astype(np.float32)

print(image.shape)

model = model.cuda()

image = tc.tensor(image[None, ...], dtype=tc.float32).cuda()

output = nn.Sigmoid()(model(image))

output[output > 0.5] = 1
output[output < 0.5] = 0

output = output.detach().cpu().numpy().astype(np.uint8)

affine = np.eye(4)
nifti_img = nib.Nifti1Image(output, affine)
nib.save(nifti_img, 'evaluation/evaluation_results/output_image.nii.gz')

In [7]:
import numpy as np
import sys
import os
import nibabel as nib
sys.path.append(os.getcwd().removesuffix('\\evaluation'))
print(sys.path)

from models.tooth_fairy_1 import build_hyspark_tooth_fairy_1
from models.class_segmentor_model import build_class_segmentor_model
import torch as tc
import torch.nn as nn
from evaluation.evaluate_agh_segmentor import evaluate_agh
from processing.utils import load_itk, crop_to_mask
from processing.utils import embed_in_center
from tqdm import tqdm

model =build_hyspark_tooth_fairy_1(in_channel=1, n_classes=1, img_size=96, freeze=False)
model.load_state_dict(tc.load('../models/weights/tooth_fairy_3d_best.pt', weights_only=True))
file = 'D:/db1_v1.0/db1_v1.0_nifti/10/image.nii.gz'
helper = 'D:/db1_v1.0/db1_v1.0_nifti_teeth_masks_multi_cls_ordered_weak/10.nii.gz'

model = model.cuda()

image = load_itk(file)
helper = load_itk(helper)

canal_mask = np.zeros(image.shape)

for teeth_num in tqdm(np.unique(helper)[1:]):

    temp_mask = np.zeros(helper.shape)

    temp_mask[helper != teeth_num] = 0
    temp_mask[helper == teeth_num] = 1 

    image_snipped, mask_snipped, coords = crop_to_mask(image, temp_mask, padding=5, input=1, coordss=True)
    
    masked = image_snipped[mask_snipped>0]
    image_snipped[mask_snipped>0] = (masked - masked.min())/(masked.max() - masked.min() + 1e-5)
    image_snipped[mask_snipped<1] = 0 

    if image_snipped.shape[0] < 96 and image_snipped.shape[1] < 96 and image.shape[2] < 96: 
        image_snip_2, cords_2 = embed_in_center(image_snipped, 96, coords=True)

        image_snip_2 = tc.tensor(image_snip_2, dtype = tc.float32).cuda()

        output = nn.Sigmoid()(model(image_snip_2)).detach().cpu().numpy()
        output[output > 0.5] = 1
        output[output < 0.5] = 0

        z_min, z_max, y_min, y_max, x_min, x_max = coords
        start, end = cords_2

        canal_mask[z_min-start[0]:z_max+96-end[0], y_min - start[1]:y_max+96-end[1], x_min - start[2]:x_max+96-end[2]] = output
     
    else:   
            image_snip_2, cords_2 = embed_in_center(image_snipped, 2*96, coords=True)
            image_snip_2 = image_snip_2[:, 96//2:96//2+96, 96//2:96//2+96]

            image_snip_2_new = np.zeros((2, 96, 96, 96))

            image_snip_2_new[0] = image_snip_2[:96]
            image_snip_2_new[1] = image_snip_2[96:]

            image_snip_2_new = tc.tensor(image_snip_2_new, dtype = tc.float32).cuda()
            image_snip_2_new = image_snip_2_new[:, None, ...]

            output = nn.Sigmoid()(model(image_snip_2_new)).detach().cpu().numpy()
            output[output > 0.5] = 1
            output[output < 0.5] = 0

            output_fin = np.zeros((2*96, 96, 96))
            output_fin[:96] = output[0]
            output_fin[96:] = output[1]

            z_min, z_max, y_min, y_max, x_min, x_max = coords
            start, end = cords_2

            canal_mask[z_min-start[0]:z_max+2*96-end[0], y_min - start[1]:y_max+96-end[1], x_min - start[2]:x_max+96-end[2]] = output


affine = np.eye(4)
nifti_img = nib.Nifti1Image(canal_mask, affine)
nib.save(nifti_img, 'evaluation/evaluation_results/output_canals_mask.nii.gz')

['c:\\Users\\Jan\\miniconda3\\python312.zip', 'c:\\Users\\Jan\\miniconda3\\DLLs', 'c:\\Users\\Jan\\miniconda3\\Lib', 'c:\\Users\\Jan\\miniconda3', '', 'c:\\Users\\Jan\\miniconda3\\Lib\\site-packages', 'c:\\Users\\Jan\\miniconda3\\Lib\\site-packages\\win32', 'c:\\Users\\Jan\\miniconda3\\Lib\\site-packages\\win32\\lib', 'c:\\Users\\Jan\\miniconda3\\Lib\\site-packages\\Pythonwin', 'c:\\Users\\Jan\\Desktop\\tooth_fairy_challenge', 'c:\\Users\\Jan\\Desktop\\tooth_fairy_challenge', 'c:\\Users\\Jan\\Desktop\\tooth_fairy_challenge', 'c:\\Users\\Jan\\Desktop\\tooth_fairy_challenge', 'c:\\Users\\Jan\\Desktop\\tooth_fairy_challenge', 'c:\\Users\\Jan\\Desktop\\tooth_fairy_challenge', 'c:\\Users\\Jan\\Desktop\\tooth_fairy_challenge']


  0%|          | 0/29 [00:07<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 432.00 MiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 11.98 GiB is allocated by PyTorch, and 153.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)